# Export quantized audio sample files

Collect test clips once, then generate `audio_samples/<model>.rs` for each `.tflite` model in `models/` whose filename does **not** contain `mel`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from utils import (
    SAMPLE_RATE,
    TARGET_AUDIO_LEN_TIME,
    DATASET_ROOT,
    REPO_ROOT,
)
from rust_export import (
    collect_test_clips_for_rs,
    make_time_feature_fn,
    write_audio_sample_quantized_rs,
    write_audio_sample_rs,
)

MODELS_DIR = REPO_ROOT / "models"
OUT_DIR = REPO_ROOT / "audio_samples"
OUT_NO_QUANTIZED_RS = OUT_DIR / "no_quantized.rs"

In [ ]:
clips = collect_test_clips_for_rs(
    DATASET_ROOT / "testing",
    sample_rate=SAMPLE_RATE,
    target_len=TARGET_AUDIO_LEN_TIME,
    num_per_label=2,
)

write_audio_sample_rs(
    OUT_NO_QUANTIZED_RS,
    clips,
    SAMPLE_RATE,
    generator_name="building_tensorflow/export_audio_sample_rs.ipynb",
)

feature_fn = make_time_feature_fn(TARGET_AUDIO_LEN_TIME)

tflite_models = sorted(
    p for p in MODELS_DIR.glob("*.tflite") if ("mel" not in p.stem.lower()) or ("sincnet" in p.stem.lower())
)

if not tflite_models:
    raise RuntimeError(f"No non-mel .tflite models found in {MODELS_DIR}")

written = []
for model_path in tflite_models:
    out_rs = OUT_DIR / f"{model_path.stem}.rs"
    write_audio_sample_quantized_rs(
        out_rs,
        clips,
        SAMPLE_RATE,
        model_path,
        feature_fn=feature_fn,
        generator_name="building_tensorflow/export_audio_sample_rs.ipynb",
    )
    written.append(out_rs)

print("Wrote non-quantized mel sample file:")
print(" -", OUT_NO_QUANTIZED_RS)
print("Generated", len(written), "quantized audio sample files:")
for p in written:
    print(" -", p)
print("clips=", len(clips), "samples_per_clip=", len(clips[0][1]))

Found 1393 files belonging to 2 classes.
Wrote non-quantized mel sample file:
 - /home/nathan/Documents/tiny-chirp-microflow/audio_samples/no_quantized.rs
Generated 4 quantized audio sample files:
 - /home/nathan/Documents/tiny-chirp-microflow/audio_samples/cnn_time_tf.rs
 - /home/nathan/Documents/tiny-chirp-microflow/audio_samples/leaf_tf.rs
 - /home/nathan/Documents/tiny-chirp-microflow/audio_samples/sincnet_multi_tf.rs
 - /home/nathan/Documents/tiny-chirp-microflow/audio_samples/sincnet_tf.rs
clips= 4 samples_per_clip= 47872


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
